# NOTICE: Production execution lives in main.py. This notebook is exploratory only; do not run it for live trading.


In [ ]:
# bot.ipynb (نسخه نهایی اصلاح‌شده)
from module.mt5 import *
from module.indicators import *
from module.modifyPosition import *
from module.stg import *
from module.state_io import save_state, load_state
import datetime  # ← اضافه شده
import time

# ----------------------------- مقداردهی اولیه MT5 ----------------------------- #
mt5.initialize()

# ----------------------------- تنظیمات کلی ربات ----------------------------- #
symbol = 'XAUUSD'
tf = '1m'
total_balance = 1000
daily_drow_down = 5
full_drow_down = 12
risk = 0.1
rr = 2
news_filter = False  # Economic-calendar integration is not implemented; do not trade around news automatically.
trend_kalman = kalman_trend_levels(symbol, tf , 50,150)
signal_kalman = Trend_change_signal_at(trend_kalman[-2])
trend_super = supertrend(symbol , tf, 10, 3 , 'ha')
signal_super = Trend_change_signal(trend_super['position'])
price = mt5.symbol_info_tick(symbol).bid
if (trend_kalman['trend'][-2] == 'long' and signal_super == 'buy') or (trend_super == 'long' and signal_kalman == 'buy' ):
    lot =lot_calculator(symbol , tf , price , sl)
    bounded_risk_percent(risk, max_risk=float(os.getenv("RISK_MAX_PERCENT", "2.0")))
# --- Central trade lifecycle integration ---
from module.state_machine import TradeRecord, TradeState, TradeStateMachine
from module.recovery import TradeRecovery
from module.observability import configure_logging, get_logger
from module.telegram_alerts import TelegramAlertNotifier
from module.memory import SecondBrain
from module.config import TRADING_MODE
from module.risk import bounded_risk_percent
from module.risk_metrics import broker_metrics
import os

trade_memory = SecondBrain("memory/events.jsonl")
trade_states = TradeStateMachine("trade_state.json", memory=trade_memory)
recovery_broker = paper_broker if TRADING_MODE == "PAPER" else mt5
trade_recovery = TradeRecovery(recovery_broker, timeout_seconds=120)
telegram_alert_notifier = TelegramAlertNotifier()
runtime_logger = configure_logging(
    os.getenv("LOG_DIR", "logs"),
    notifier=telegram_alert_notifier if telegram_alert_notifier.enabled else None,
)
logger = get_logger("runtime")

def managed_create_order(symbol, lot, order_type, sl=0.0, tp=0.0, comment="hashem", tf="unknown"):
    """Create one signal, validate it, execute through PAPER/LIVE guard, and persist state."""
    tick = mt5.symbol_info_tick(symbol)
    candle_time = getattr(tick, "time", 0) if tick is not None else 0
    trade_id = f"{symbol}:{tf}:{candle_time}:{comment}:{order_type}"
    try:
        trade_states.create(TradeRecord(
            trade_id=trade_id,
            symbol=symbol,
            strategy=comment,
            candle_time=str(candle_time),
        ))
        trade_states.transition(trade_id, TradeState.VALIDATED)
        trade_states.transition(trade_id, TradeState.APPROVED)
        daily_profit, open_positions = broker_metrics(
            mt5, paper_broker if TRADING_MODE == "PAPER" else None, TRADING_MODE
        )
        result = create_order(
            symbol, lot, order_type, sl, tp, comment, trade_id=trade_id,
            daily_profit=daily_profit, open_positions=open_positions
        )
        trade_states.transition(trade_id, TradeState.SENT)
        if result is not None and result.retcode == mt5.TRADE_RETCODE_DONE:
            trade_states.transition(
                trade_id,
                TradeState.ACCEPTED,
                ticket=getattr(result, "order", 0),
            )
            return result
        trade_states.transition(trade_id, TradeState.REJECTED, reason="MT5 execution failed")
        return result
    except (ValueError, RuntimeError, KeyError) as exc:
        record = trade_states.get(trade_id)
        if record is not None and record.state not in {TradeState.CLOSED, TradeState.REJECTED}:
            trade_states.transition(trade_id, TradeState.REJECTED, reason=str(exc))
        logger.exception("trade rejected", extra={"event": "trade_rejected", "trade_id": trade_id})
        return None

def reconcile_trade_state(trade_id, is_open, is_managed=False):
    """Advance persisted state only after observing the real MT5 position."""
    record = trade_states.get(trade_id)
    if record is None:
        return None
    if is_open and record.state == TradeState.ACCEPTED:
        trade_states.transition(trade_id, TradeState.OPEN)
    if is_open and is_managed and trade_states.get(trade_id).state in {TradeState.OPEN, TradeState.MANAGED}:
        trade_states.transition(trade_id, TradeState.MANAGED)
    if not is_open and trade_states.get(trade_id).state in {TradeState.OPEN, TradeState.MANAGED}:
        trade_states.transition(trade_id, TradeState.CLOSED)
    return trade_states.get(trade_id)


def reconcile_all_trade_states():
    """Reconcile persisted trade states with currently visible MT5 positions."""
    recovery_report = trade_recovery.recover(trade_states)
    if recovery_report.errors:
        logger.error("Recovery errors", extra={"event": "recovery_errors"})
    for trade_id, record in list(trade_states.records.items()):
        if record.state in {TradeState.CLOSED, TradeState.REJECTED, TradeState.SIGNAL, TradeState.VALIDATED, TradeState.APPROVED, TradeState.SENT}:
            continue
        try:
            positions = mt5.positions_get(symbol=record.symbol) or ()
        except Exception as exc:
            print(f"State reconciliation failed for {trade_id}: {exc}")
            continue

        matching = []
        for position in positions:
            same_magic = getattr(position, "magic", MAGIC_NUMBER) == MAGIC_NUMBER
            same_ticket = bool(record.ticket and getattr(position, "ticket", 0) == record.ticket)
            same_strategy = getattr(position, "comment", "") == record.strategy
            if same_magic and (same_ticket or (same_strategy and getattr(position, "symbol", "") == record.symbol)):
                matching.append(position)

        if matching:
            position = matching[0]
            if record.state == TradeState.ACCEPTED:
                reconcile_trade_state(trade_id, is_open=True)
            current = trade_states.get(trade_id)
            if current is not None and current.state in {TradeState.OPEN, TradeState.MANAGED}:
                reconcile_trade_state(trade_id, is_open=True, is_managed=True)
        elif record.state in {TradeState.OPEN, TradeState.MANAGED}:
            reconcile_trade_state(trade_id, is_open=False)


In [ ]:
def supertrend_stg(symbol , tf, atr_period=10, multiplier=3.0 , candle_type = 'ha' ,lot = 0.01 , rr = 2 , comment = 'hashem'):
    trend = supertrend(symbol , tf, atr_period, multiplier , candle_type)
    signal = Trend_change_signal(trend['position'])
    price = mt5.symbol_info_tick(symbol)

    if signal == 'buy' :
        sl = trend['value'][-1]
        tp = price.ask + ((price.ask - sl )* rr)
        managed_create_order(symbol , lot , buy , sl , tp , comment)

    if signal == 'sell' :
        sl = trend['value'][-1]
        tp = price.bid - ((sl - price.bid)* rr)
        managed_create_order(symbol , lot , sell , sl , tp , comment)

In [ ]:
# bot.ipynb
from module.mt5 import *
from module.indicators import *
from module.modifyPosition import *
from module.stg import *
from module.state_io import save_state, load_state
import MetaTrader5 as mt5
import time
import datetime

mt5.initialize()

# --------------------------- تنظیمات اصلی --------------------------- #
symbol = 'XAUUSD'
tf = '1m'
Start_balance = balance()
risk = 1
rr = 2


# ======================================================================================== #
# استراتژی 1: ADX + SuperTrend
# نکته: ADX فقط قدرت روند می‌دهد نه جهت، پس از ADX>25 به عنوان فیلتر قدرت استفاده می‌کنیم
# ======================================================================================== #
def adx_super_stg(symbol, tf, candle_type='ha', rr=2, comment='adx_super_stg'):

    # مدیریت پوزیشن‌های باز
    positions = mt5.positions_get(symbol=symbol)
    if positions:
        for position in positions:
            if position.comment == comment:
                tp1_save_profit(position)
                smartTP(tf, position)

    # جلوگیری از چند ورود
    if total_position_comment(comment) > 0:
        return

    # قیمت
    price = mt5.symbol_info_tick(symbol)
    if price is None:
        return

    # سیگنال‌ها
    adx_val = adx(symbol, tf, 14, 14, candle_type)
    trend_super = supertrend(symbol, tf, 10, 3, candle_type)
    super_pos_list = trend_super['position']
    signal_super = Trend_change_signal(super_pos_list)

    # شرط ورود: ADX بالای 25 (روند قوی) + سیگنال سوپرترند
    # buy
    if adx_val[-2] > 25 and super_pos_list[-2] == 'long' and signal_super == 'buy':
        sl = price.ask - smartSL(symbol, tf, 7, 4)
        tp = price.ask + ((price.ask - sl) * rr)
        org_risk = bounded_risk_percent(risk, max_risk=float(os.getenv("RISK_MAX_PERCENT", "2.0")))
        lot = lot_calculator(symbol, org_risk, price.ask, sl)
        managed_create_order(symbol, lot, buy, sl, tp, comment)

    # sell
    if adx_val[-2] > 25 and super_pos_list[-2] == 'short' and signal_super == 'sell':
        sl = price.bid + smartSL(symbol, tf, 7, 4)
        tp = price.bid - ((sl - price.bid) * rr)
        org_risk = bounded_risk_percent(risk, max_risk=float(os.getenv("RISK_MAX_PERCENT", "2.0")))
        lot = lot_calculator(symbol, org_risk, price.bid, sl)
        managed_create_order(symbol, lot, sell, sl, tp, comment)


# ======================================================================================== #
# استراتژی 2: Ichimoku + SuperTrend
# روند ایچیموکو: cloud = green یعنی long، cloud = red یعنی short
# ======================================================================================== #
def ichimoku_super_stg(symbol, tf, candle_type='ha', rr=2, comment='ichimoku_super_stg'):

    # مدیریت پوزیشن‌های باز
    positions = mt5.positions_get(symbol=symbol)
    if positions:
        for position in positions:
            if position.comment == comment:
                tp1_save_profit(position)
                smartTP(tf, position)

    # جلوگیری از چند ورود
    if total_position_comment(comment) > 0:
        return

    # قیمت
    price = mt5.symbol_info_tick(symbol)
    if price is None:
        return

    # سیگنال‌ها
    ichi = ichimoku(symbol, tf, 9, 26, 26, 52, candle_type)
    ichi_cloud = ichi['cloud']
    trend_super = supertrend(symbol, tf, 10, 3, candle_type)
    super_pos_list = trend_super['position']
    signal_super = Trend_change_signal(super_pos_list)

    # buy: ابر سبز + سیگنال سوپر buy
    if ichi_cloud[-2] == 'green' and signal_super == 'buy':
        sl = price.ask - smartSL(symbol, tf, 7, 4)
        tp = price.ask + ((price.ask - sl) * rr)
        org_risk = bounded_risk_percent(risk, max_risk=float(os.getenv("RISK_MAX_PERCENT", "2.0")))
        lot = lot_calculator(symbol, org_risk, price.ask, sl)
        managed_create_order(symbol, lot, buy, sl, tp, comment)

    # sell: ابر قرمز + سیگنال سوپر sell
    if ichi_cloud[-2] == 'red' and signal_super == 'sell':
        sl = price.bid + smartSL(symbol, tf, 7, 4)
        tp = price.bid - ((sl - price.bid) * rr)
        org_risk = bounded_risk_percent(risk, max_risk=float(os.getenv("RISK_MAX_PERCENT", "2.0")))
        lot = lot_calculator(symbol, org_risk, price.bid, sl)
        managed_create_order(symbol, lot, sell, sl, tp, comment)


# ======================================================================================== #
# استراتژی 3: SuperTrend + Parabolic SAR
# ======================================================================================== #
def super_sar_stg(symbol, tf, candle_type='ha', rr=2, comment='super_sar_stg'):

    # مدیریت پوزیشن‌های باز
    positions = mt5.positions_get(symbol=symbol)
    if positions:
        for position in positions:
            if position.comment == comment:
                tp1_save_profit(position)
                smartTP(tf, position)

    # جلوگیری از چند ورود
    if total_position_comment(comment) > 0:
        return

    # قیمت
    price = mt5.symbol_info_tick(symbol)
    if price is None:
        return

    # سیگنال‌ها
    trend_super = supertrend(symbol, tf, 10, 3, candle_type)
    super_pos_list = trend_super['position']
    signal_sar = sar_signal(symbol, tf, 0.02, 0.02, 0.2, candle_type)

    # buy: سوپر long + سار buy
    if super_pos_list[-2] == 'long' and signal_sar == 'buy':
        sl = price.ask - smartSL(symbol, tf, 7, 4)
        tp = price.ask + ((price.ask - sl) * rr)
        org_risk = bounded_risk_percent(risk, max_risk=float(os.getenv("RISK_MAX_PERCENT", "2.0")))
        lot = lot_calculator(symbol, org_risk, price.ask, sl)
        managed_create_order(symbol, lot, buy, sl, tp, comment)

    # sell: سوپر short + سار sell
    if super_pos_list[-2] == 'short' and signal_sar == 'sell':
        sl = price.bid + smartSL(symbol, tf, 7, 4)
        tp = price.bid - ((sl - price.bid) * rr)
        org_risk = bounded_risk_percent(risk, max_risk=float(os.getenv("RISK_MAX_PERCENT", "2.0")))
        lot = lot_calculator(symbol, org_risk, price.bid, sl)
        managed_create_order(symbol, lot, sell, sl, tp, comment)


# ======================================================================================== #
# استراتژی 4: SuperTrend + MACD
# روند MACD: histogram > 0 = long، < 0 = short
# سیگنال: تغییر جهت histogram
# ======================================================================================== #
def super_macd_stg(symbol, tf, candle_type='ha', rr=2, comment='super_macd_stg'):

    # مدیریت پوزیشن‌های باز
    positions = mt5.positions_get(symbol=symbol)
    if positions:
        for position in positions:
            if position.comment == comment:
                tp1_save_profit(position)
                smartTP(tf, position)

    # جلوگیری از چند ورود
    if total_position_comment(comment) > 0:
        return

    # قیمت
    price = mt5.symbol_info_tick(symbol)
    if price is None:
        return

    # سیگنال‌ها
    trend_super = supertrend(symbol, tf, 10, 3, candle_type)
    super_pos_list = trend_super['position']
    macd_data = macd(symbol, tf, 12, 26, 9)
    hist = macd_data['histogram']

    # سیگنال MACD: تغییر جهت histogram
    macd_buy = hist[-3] < 0 and hist[-2] > 0
    macd_sell = hist[-3] > 0 and hist[-2] < 0

    # buy: سوپر long + MACD buy
    if super_pos_list[-2] == 'long' and macd_buy:
        sl = price.ask - smartSL(symbol, tf, 7, 4)
        tp = price.ask + ((price.ask - sl) * rr)
        org_risk = bounded_risk_percent(risk, max_risk=float(os.getenv("RISK_MAX_PERCENT", "2.0")))
        lot = lot_calculator(symbol, org_risk, price.ask, sl)
        managed_create_order(symbol, lot, buy, sl, tp, comment)

    # sell: سوپر short + MACD sell
    if super_pos_list[-2] == 'short' and macd_sell:
        sl = price.bid + smartSL(symbol, tf, 7, 4)
        tp = price.bid - ((sl - price.bid) * rr)
        org_risk = bounded_risk_percent(risk, max_risk=float(os.getenv("RISK_MAX_PERCENT", "2.0")))
        lot = lot_calculator(symbol, org_risk, price.bid, sl)
        managed_create_order(symbol, lot, sell, sl, tp, comment)


# ======================================================================================== #
# استراتژی 5: SuperTrend + Keltner Channels
# روند کلتنر: close > upper = long، close < lower = short
# ======================================================================================== #
def super_keltner_stg(symbol, tf, candle_type='ha', rr=2, comment='super_keltner_stg'):

    # مدیریت پوزیشن‌های باز
    positions = mt5.positions_get(symbol=symbol)
    if positions:
        for position in positions:
            if position.comment == comment:
                tp1_save_profit(position)
                smartTP(tf, position)

    # جلوگیری از چند ورود
    if total_position_comment(comment) > 0:
        return

    # قیمت
    price = mt5.symbol_info_tick(symbol)
    if price is None:
        return

    # کندل‌ها برای چک کردن close
    candles = candle(symbol, tf, 5)

    # سیگنال‌ها
    trend_super = supertrend(symbol, tf, 10, 3, candle_type)
    super_pos_list = trend_super['position']
    signal_super = Trend_change_signal(super_pos_list)
    keltner_up = keltner_channels(symbol, tf, 'up', 20, 10, 2, candle_type)
    keltner_low = keltner_channels(symbol, tf, 'low', 20, 10, 2, candle_type)

    # روند کلتنر
    keltner_long = candles[-2]['close'] > keltner_up[-2]
    keltner_short = candles[-2]['close'] < keltner_low[-2]

    # buy: کلتنر long + سوپر buy
    if keltner_long and signal_super == 'buy':
        sl = price.ask - smartSL(symbol, tf, 7, 4)
        tp = price.ask + ((price.ask - sl) * rr)
        org_risk = bounded_risk_percent(risk, max_risk=float(os.getenv("RISK_MAX_PERCENT", "2.0")))
        lot = lot_calculator(symbol, org_risk, price.ask, sl)
        managed_create_order(symbol, lot, buy, sl, tp, comment)

    # sell: کلتنر short + سوپر sell
    if keltner_short and signal_super == 'sell':
        sl = price.bid + smartSL(symbol, tf, 7, 4)
        tp = price.bid - ((sl - price.bid) * rr)
        org_risk = bounded_risk_percent(risk, max_risk=float(os.getenv("RISK_MAX_PERCENT", "2.0")))
        lot = lot_calculator(symbol, org_risk, price.bid, sl)
        managed_create_order(symbol, lot, sell, sl, tp, comment)


# ======================================================================================== #
# استراتژی 6: SuperTrend + Trend Magic
# ======================================================================================== #
def super_trendmagic_stg(symbol, tf, candle_type='ha', rr=2, comment='super_trendmagic_stg'):

    # مدیریت پوزیشن‌های باز
    positions = mt5.positions_get(symbol=symbol)
    if positions:
        for position in positions:
            if position.comment == comment:
                tp1_save_profit(position)
                smartTP(tf, position)

    # جلوگیری از چند ورود
    if total_position_comment(comment) > 0:
        return

    # قیمت
    price = mt5.symbol_info_tick(symbol)
    if price is None:
        return

    # سیگنال‌ها
    trend_super = supertrend(symbol, tf, 10, 3, candle_type)
    super_pos_list = trend_super['position']
    signal_super = Trend_change_signal(super_pos_list)
    tm = trend_magic(symbol, tf, 20, 5, 1, candle_type)
    tm_signal_list = tm['signal']

    # buy: ترند مجیک long + سوپر buy
    if tm_signal_list[-2] == 'long' and signal_super == 'buy':
        sl = price.ask - smartSL(symbol, tf, 7, 4)
        tp = price.ask + ((price.ask - sl) * rr)
        org_risk = bounded_risk_percent(risk, max_risk=float(os.getenv("RISK_MAX_PERCENT", "2.0")))
        lot = lot_calculator(symbol, org_risk, price.ask, sl)
        managed_create_order(symbol, lot, buy, sl, tp, comment)

    # sell: ترند مجیک short + سوپر sell
    if tm_signal_list[-2] == 'short' and signal_super == 'sell':
        sl = price.bid + smartSL(symbol, tf, 7, 4)
        tp = price.bid - ((sl - price.bid) * rr)
        org_risk = bounded_risk_percent(risk, max_risk=float(os.getenv("RISK_MAX_PERCENT", "2.0")))
        lot = lot_calculator(symbol, org_risk, price.bid, sl)
        managed_create_order(symbol, lot, sell, sl, tp, comment)


# ======================================================================================== #
# استراتژی 7: SuperTrend + Volumatic VIDYA
# ======================================================================================== #
def super_vidya_stg(symbol, tf, candle_type='ha', rr=2, comment='super_vidya_stg'):

    # مدیریت پوزیشن‌های باز
    positions = mt5.positions_get(symbol=symbol)
    if positions:
        for position in positions:
            if position.comment == comment:
                tp1_save_profit(position)
                smartTP(tf, position)

    # جلوگیری از چند ورود
    if total_position_comment(comment) > 0:
        return

    # قیمت
    price = mt5.symbol_info_tick(symbol)
    if price is None:
        return

    # سیگنال‌ها
    trend_super = supertrend(symbol, tf, 10, 3, candle_type)
    super_pos_list = trend_super['position']
    signal_super = Trend_change_signal(super_pos_list)
    vidya_trend = volumatic_vidya(symbol, tf, 10, 20, 2, 'trend', candle_type)

    # buy: ویدیا long + سوپر buy
    if vidya_trend[-2] == 'long' and signal_super == 'buy':
        sl = price.ask - smartSL(symbol, tf, 7, 4)
        tp = price.ask + ((price.ask - sl) * rr)
        org_risk = bounded_risk_percent(risk, max_risk=float(os.getenv("RISK_MAX_PERCENT", "2.0")))
        lot = lot_calculator(symbol, org_risk, price.ask, sl)
        managed_create_order(symbol, lot, buy, sl, tp, comment)

    # sell: ویدیا short + سوپر sell
    if vidya_trend[-2] == 'short' and signal_super == 'sell':
        sl = price.bid + smartSL(symbol, tf, 7, 4)
        tp = price.bid - ((sl - price.bid) * rr)
        org_risk = bounded_risk_percent(risk, max_risk=float(os.getenv("RISK_MAX_PERCENT", "2.0")))
        lot = lot_calculator(symbol, org_risk, price.bid, sl)
        managed_create_order(symbol, lot, sell, sl, tp, comment)


# ======================================================================================== #
# استراتژی 8: SuperTrend + Trend Ali
# ======================================================================================== #
def super_trendali_stg(symbol, tf, candle_type='ha', rr=2, comment='super_trendali_stg'):

    # مدیریت پوزیشن‌های باز
    positions = mt5.positions_get(symbol=symbol)
    if positions:
        for position in positions:
            if position.comment == comment:
                tp1_save_profit(position)
                smartTP(tf, position)

    # جلوگیری از چند ورود
    if total_position_comment(comment) > 0:
        return

    # قیمت
    price = mt5.symbol_info_tick(symbol)
    if price is None:
        return

    # سیگنال‌ها
    trend_super = supertrend(symbol, tf, 10, 3, candle_type)
    super_pos_list = trend_super['position']
    signal_super = Trend_change_signal(super_pos_list)
    tali = trend_ali(symbol, tf, 60, 6, 'Hma', candle_type)
    tali_trend = tali['trend']

    # buy: ترند علی long + سوپر buy
    if tali_trend[-2] == 'long' and signal_super == 'buy':
        sl = price.ask - smartSL(symbol, tf, 7, 4)
        tp = price.ask + ((price.ask - sl) * rr)
        org_risk = bounded_risk_percent(risk, max_risk=float(os.getenv("RISK_MAX_PERCENT", "2.0")))
        lot = lot_calculator(symbol, org_risk, price.ask, sl)
        managed_create_order(symbol, lot, buy, sl, tp, comment)

    # sell: ترند علی short + سوپر sell
    if tali_trend[-2] == 'short' and signal_super == 'sell':
        sl = price.bid + smartSL(symbol, tf, 7, 4)
        tp = price.bid - ((sl - price.bid) * rr)
        org_risk = bounded_risk_percent(risk, max_risk=float(os.getenv("RISK_MAX_PERCENT", "2.0")))
        lot = lot_calculator(symbol, org_risk, price.bid, sl)
        managed_create_order(symbol, lot, sell, sl, tp, comment)


# ======================================================================================== #
# اجرای ربات
# ======================================================================================== #
while True:
    reconcile_all_trade_states()
    try:
        mt5.symbol_select(symbol, True)

        # اجرای همه استراتژی‌ها
        # هر تابع خودش چک می‌کند که پوزیشن باز دارد یا نه
        adx_super_stg(symbol, tf, 'ha', rr, 'adx_super_stg')
        ichimoku_super_stg(symbol, tf, 'ha', rr, 'ichimoku_super_stg')
        super_sar_stg(symbol, tf, 'ha', rr, 'super_sar_stg')
        super_macd_stg(symbol, tf, 'ha', rr, 'super_macd_stg')
        super_keltner_stg(symbol, tf, 'ha', rr, 'super_keltner_stg')
        super_trendmagic_stg(symbol, tf, 'ha', rr, 'super_trendmagic_stg')
        super_vidya_stg(symbol, tf, 'ha', rr, 'super_vidya_stg')
        super_trendali_stg(symbol, tf, 'ha', rr, 'super_trendali_stg')

        time.sleep(1)

    except Exception as e:
        print(e)
        time.sleep(10)
        continue
    
from module.execution import ExecutionEngine
execution_engine = ExecutionEngine(mt5)


In [1]:
"""
XAUUSD Hedge Grid Basket EA - Python / MetaTrader 5

منطق:
- هر سیکل: 5 Buy Stop بالای قیمت و 5 Sell Stop پایین قیمت
- فاصله پیش‌فرض هر Level: 0.20 قیمت طلا (قابل تنظیم)
- اگر 5 معامله Buy فعال شوند -> بستن همه معاملات + حذف همه Pendingها
- اگر 5 معامله Sell فعال شوند -> بستن همه معاملات + حذف همه Pendingها
- اگر سود خالص Basket به هدف برسد -> بستن همه + حذف Pendingها
- پس از پاک‌سازی کامل، 1 ثانیه صبر و سیکل جدید
- فقط معاملات/سفارش‌های دارای Magic Number خود ربات مدیریت می‌شوند

پیش‌نیاز:
    pip install MetaTrader5

قبل از اجرا:
1) MetaTrader 5 را باز و به حساب معاملاتی وصل کنید.
2) Algo Trading/اجازه معامله خودکار را فعال کنید.
3) SYMBOL را دقیقاً مطابق نام نماد بروکر تنظیم کنید؛ ممکن است XAUUSDm یا GOLD باشد.
4) فاصله GRID_STEP_PRICE و BASKET_TARGET_USD را بررسی کنید.
"""

import time
from dataclasses import dataclass
from typing import Optional

import MetaTrader5 as mt5


# =========================
# CONFIG
# =========================

SYMBOL = "XAUUSD"

MAGIC = 26080901

LOTS = 0.01

# برای طلا، این مقدار فاصله قیمتی است، نه "pip" استاندارد فارکس.
# مثال: 0.20 یعنی:
# 4334.20 -> 4334.40
GRID_STEP_PRICE = 0.20

GRID_LEVELS = 5

# سود خالص کل Basket به دلار حساب
# وقتی Profit + Swap + Commission به این مقدار برسد،
# کل سیکل بسته می‌شود.
BASKET_TARGET_USD = 1.0

# در صورت فعال شدن هر 5 معامله در یک سمت، کل سیکل بسته می‌شود.
CLOSE_ON_FULL_SIDE = True

RESTART_DELAY_SECONDS = 1.0

# فاصله زمانی بررسی بازار
LOOP_INTERVAL_SECONDS = 0.20

# اگر True باشد، هنگام شروع برنامه، سفارش‌های قدیمی همین Magic پاک می‌شوند.
CLEAN_OLD_ORDERS_ON_START = True


# =========================
# DATA
# =========================

@dataclass
class CycleState:
    active: bool = False
    cycle_id: int = 0


cycle = CycleState()


# =========================
# MT5 HELPERS
# =========================

def log(msg: str):
    print(time.strftime("[%Y-%m-%d %H:%M:%S]"), msg, flush=True)


def initialize_mt5() -> bool:
    if not mt5.initialize():
        log(f"MT5 initialize failed: {mt5.last_error()}")
        return False

    info = mt5.symbol_info(SYMBOL)
    if info is None:
        log(f"Symbol not found: {SYMBOL}")
        return False

    if not info.visible:
        if not mt5.symbol_select(SYMBOL, True):
            log(f"Could not select symbol: {SYMBOL}")
            return False

    log(f"Connected to MT5 | {SYMBOL}")
    return True


def symbol_info():
    return mt5.symbol_info(SYMBOL)


def tick():
    return mt5.symbol_info_tick(SYMBOL)


def normalize_price(price: float) -> float:
    info = symbol_info()
    return round(price, info.digits)


def normalize_volume(volume: float) -> float:
    info = symbol_info()

    step = info.volume_step
    min_vol = info.volume_min
    max_vol = info.volume_max

    volume = max(min_vol, min(max_vol, volume))

    steps = round(volume / step)
    volume = steps * step

    # جلوگیری از خطای floating point
    return round(volume, 8)


def filling_mode():
    info = symbol_info()

    # برای اکثر بروکرها ORDER_FILLING_RETURN قابل استفاده است،
    # ولی بعضی نمادها فقط FOK/IOC را می‌پذیرند.
    # ابتدا RETURN را امتحان می‌کنیم.
    return mt5.ORDER_FILLING_RETURN


# =========================
# ORDERS / POSITIONS
# =========================

def get_our_positions():
    positions = mt5.positions_get(symbol=SYMBOL)

    if positions is None:
        return []

    return [
        p for p in positions
        if p.magic == MAGIC
    ]


def get_our_pending_orders():
    orders = mt5.orders_get(symbol=SYMBOL)

    if orders is None:
        return []

    pending_types = {
        mt5.ORDER_TYPE_BUY_LIMIT,
        mt5.ORDER_TYPE_SELL_LIMIT,
        mt5.ORDER_TYPE_BUY_STOP,
        mt5.ORDER_TYPE_SELL_STOP,
        mt5.ORDER_TYPE_BUY_STOP_LIMIT,
        mt5.ORDER_TYPE_SELL_STOP_LIMIT,
    }

    return [
        o for o in orders
        if o.magic == MAGIC and o.type in pending_types
    ]


def basket_profit() -> float:
    """
    سود/زیان فعلی معاملات باز ربات:
    profit + swap + commission
    """
    positions = get_our_positions()

    total = 0.0

    for p in positions:
        total += float(p.profit)
        total += float(p.swap)

        # بعضی نسخه‌ها commission را در position ارائه می‌کنند،
        # بعضی حساب‌ها ممکن است آن را صفر نشان دهند.
        total += float(getattr(p, "commission", 0.0))

    return total


def count_side_positions():
    positions = get_our_positions()

    buy_count = sum(
        1 for p in positions
        if p.type == mt5.POSITION_TYPE_BUY
    )

    sell_count = sum(
        1 for p in positions
        if p.type == mt5.POSITION_TYPE_SELL
    )

    return buy_count, sell_count


# =========================
# SEND PENDING ORDERS
# =========================

def place_buy_stop(price: float) -> bool:
    info = symbol_info()

    price = normalize_price(price)
    volume = normalize_volume(LOTS)

    request = {
        "action": mt5.TRADE_ACTION_PENDING,
        "symbol": SYMBOL,
        "volume": volume,
        "type": mt5.ORDER_TYPE_BUY_STOP,
        "price": price,
        "sl": 0.0,
        "tp": 0.0,
        "deviation": 20,
        "magic": MAGIC,
        "comment": f"HedgeGrid BUY {cycle.cycle_id}",
        "type_time": mt5.ORDER_TIME_GTC,
        "type_filling": filling_mode(),
    }

    result = execution_engine.send(request)

    if result is None:
        log(f"BUY STOP failed: {mt5.last_error()}")
        return False

    if result.retcode != mt5.TRADE_RETCODE_DONE:
        log(f"BUY STOP failed | retcode={result.retcode} | {result.comment}")
        return False

    log(f"BUY STOP placed @ {price}")
    return True


def place_sell_stop(price: float) -> bool:
    price = normalize_price(price)
    volume = normalize_volume(LOTS)

    request = {
        "action": mt5.TRADE_ACTION_PENDING,
        "symbol": SYMBOL,
        "volume": volume,
        "type": mt5.ORDER_TYPE_SELL_STOP,
        "price": price,
        "sl": 0.0,
        "tp": 0.0,
        "deviation": 20,
        "magic": MAGIC,
        "comment": f"HedgeGrid SELL {cycle.cycle_id}",
        "type_time": mt5.ORDER_TIME_GTC,
        "type_filling": filling_mode(),
    }

    result = execution_engine.send(request)

    if result is None:
        log(f"SELL STOP failed: {mt5.last_error()}")
        return False

    if result.retcode != mt5.TRADE_RETCODE_DONE:
        log(f"SELL STOP failed | retcode={result.retcode} | {result.comment}")
        return False

    log(f"SELL STOP placed @ {price}")
    return True


# =========================
# CREATE NEW CYCLE
# =========================

def create_cycle() -> bool:
    global cycle

    current_tick = tick()

    if current_tick is None:
        log("No tick available.")
        return False

    # برای جلوگیری از ایجاد سفارش داخل spread/در قیمت نامعتبر،
    # از Ask برای Buy Stop و Bid برای Sell Stop استفاده می‌کنیم.
    ask = current_tick.ask
    bid = current_tick.bid

    info = symbol_info()

    min_distance = info.trade_stops_level * info.point

    step = GRID_STEP_PRICE

    # اگر حداقل فاصله بروکر بزرگ‌تر از Grid Step باشد،
    # سفارش‌های نزدیک ممکن است reject شوند.
    if min_distance > 0 and step < min_distance:
        log(
            f"WARNING: broker minimum stop distance={min_distance}, "
            f"but GRID_STEP_PRICE={step}"
        )

    cycle.cycle_id += 1

    log(
        f"Creating cycle #{cycle.cycle_id} | "
        f"Ask={ask} Bid={bid} Step={step}"
    )

    success_buy = 0
    success_sell = 0

    # Buy Stopها بالای Ask
    for i in range(1, GRID_LEVELS + 1):
        price = ask + (step * i)

        if place_buy_stop(price):
            success_buy += 1

    # Sell Stopها پایین Bid
    for i in range(1, GRID_LEVELS + 1):
        price = bid - (step * i)

        if place_sell_stop(price):
            success_sell += 1

    if success_buy == GRID_LEVELS and success_sell == GRID_LEVELS:
        cycle.active = True
        log("New cycle ACTIVE: 5 BUY STOP + 5 SELL STOP")
        return True

    log(
        f"Cycle creation incomplete: "
        f"BUY={success_buy}/{GRID_LEVELS}, "
        f"SELL={success_sell}/{GRID_LEVELS}"
    )

    # اگر همه سفارش‌ها ساخته نشده‌اند، هر چیزی که ساخته شده پاک می‌شود.
    cleanup_cycle()

    return False


# =========================
# CLOSE / DELETE
# =========================

def delete_pending_order(order) -> bool:
    request = {
        "action": mt5.TRADE_ACTION_REMOVE,
        "order": order.ticket,
        "symbol": order.symbol,
        "magic": MAGIC,
        "comment": "HedgeGrid delete pending",
    }

    result = execution_engine.send(request)

    if result is None:
        log(f"Remove order {order.ticket} failed: {mt5.last_error()}")
        return False

    if result.retcode != mt5.TRADE_RETCODE_DONE:
        log(
            f"Remove order {order.ticket} failed | "
            f"retcode={result.retcode} | {result.comment}"
        )
        return False

    log(f"Pending deleted: {order.ticket}")
    return True


def close_position(position) -> bool:
    current_tick = tick()

    if current_tick is None:
        return False

    if position.type == mt5.POSITION_TYPE_BUY:
        order_type = mt5.ORDER_TYPE_SELL
        price = current_tick.bid
    else:
        order_type = mt5.ORDER_TYPE_BUY
        price = current_tick.ask

    request = {
        "action": mt5.TRADE_ACTION_DEAL,
        "symbol": position.symbol,
        "volume": position.volume,
        "type": order_type,
        "position": position.ticket,
        "price": price,
        "deviation": 30,
        "magic": MAGIC,
        "comment": "HedgeGrid close basket",
        "type_time": mt5.ORDER_TIME_GTC,
        "type_filling": filling_mode(),
    }

    result = execution_engine.send(request)

    if result is None:
        log(f"Close position {position.ticket} failed: {mt5.last_error()}")
        return False

    if result.retcode != mt5.TRADE_RETCODE_DONE:
        log(
            f"Close position {position.ticket} failed | "
            f"retcode={result.retcode} | {result.comment}"
        )
        return False

    log(f"Position closed: {position.ticket}")
    return True


def cleanup_cycle():
    """
    همه Pendingهای ربات را حذف می‌کند
    و همه Positionهای ربات را می‌بندد.
    """

    global cycle

    log("CLEANUP: closing positions and deleting pending orders...")

    # چند بار تلاش می‌کنیم چون ممکن است در همان لحظه
    # سفارش Pending تبدیل به Position شده باشد.
    for _ in range(5):

        positions = get_our_positions()

        for position in positions:
            close_position(position)

        orders = get_our_pending_orders()

        for order in orders:
            delete_pending_order(order)

        time.sleep(0.10)

        if not get_our_positions() and not get_our_pending_orders():
            break

    remaining_positions = get_our_positions()
    remaining_orders = get_our_pending_orders()

    if remaining_positions or remaining_orders:
        log(
            f"WARNING: cleanup incomplete | "
            f"positions={len(remaining_positions)} "
            f"pending={len(remaining_orders)}"
        )
        return False

    cycle.active = False
    log("CLEANUP COMPLETE: basket is empty.")
    return True


# =========================
# STARTUP CLEANUP
# =========================

def clean_old_orders():
    """
    فقط Pendingها و Positionهای دارای MAGIC خود ربات را پاک می‌کند.
    """

    if not CLEAN_OLD_ORDERS_ON_START:
        return

    positions = get_our_positions()

    if positions:
        log(
            f"WARNING: {len(positions)} old position(s) "
            f"with MAGIC={MAGIC} found."
        )

    orders = get_our_pending_orders()

    if orders:
        log(
            f"Removing {len(orders)} old pending order(s) "
            f"with MAGIC={MAGIC}."
        )

    # عمداً Positionهای قدیمی را خودکار نمی‌بندیم.
    # برای جلوگیری از بستن ناخواسته معاملات قبلی.
    for order in orders:
        delete_pending_order(order)


# =========================
# STRATEGY LOGIC
# =========================

def check_exit_conditions() -> Optional[str]:

    positions = get_our_positions()

    if not positions:
        return None

    buy_count, sell_count = count_side_positions()
    net = basket_profit()

    # شرط 1: Basket Profit
    if net >= BASKET_TARGET_USD:
        return (
            f"BASKET TARGET reached | "
            f"Net={net:.2f} >= Target={BASKET_TARGET_USD:.2f}"
        )

    # شرط 2: تمام شدن یک طرف
    if CLOSE_ON_FULL_SIDE:
        if buy_count >= GRID_LEVELS:
            return (
                f"ALL BUY LEVELS ACTIVE | "
                f"BUY={buy_count}/{GRID_LEVELS}"
            )

        if sell_count >= GRID_LEVELS:
            return (
                f"ALL SELL LEVELS ACTIVE | "
                f"SELL={sell_count}/{GRID_LEVELS}"
            )

    return None


def strategy_loop():
    global cycle

    while True:
        reconcile_all_trade_states()

        # اگر سیکل فعال نیست و هیچ چیزی متعلق به ربات وجود ندارد،
        # سیکل جدید بساز.
        if not cycle.active:

            if get_our_positions() or get_our_pending_orders():
                cleanup_cycle()
                time.sleep(RESTART_DELAY_SECONDS)
                continue

            if create_cycle():
                time.sleep(LOOP_INTERVAL_SECONDS)
                continue

            time.sleep(1.0)
            continue

        # وضعیت Basket
        positions = get_our_positions()
        orders = get_our_pending_orders()

        buy_count, sell_count = count_side_positions()
        net = basket_profit()

        log(
            f"STATUS | "
            f"BUY={buy_count} SELL={sell_count} "
            f"OPEN={len(positions)} "
            f"PENDING={len(orders)} "
            f"NET={net:.2f}"
        )

        reason = check_exit_conditions()

        if reason:
            log(f"EXIT SIGNAL: {reason}")

            if cleanup_cycle():
                log(
                    f"Cycle #{cycle.cycle_id} finished. "
                    f"Restarting in {RESTART_DELAY_SECONDS} second(s)."
                )

                time.sleep(RESTART_DELAY_SECONDS)

                # سیکل جدید در ابتدای loop ساخته می‌شود.
                continue

        time.sleep(LOOP_INTERVAL_SECONDS)


# =========================
# MAIN
# =========================

def main():

    if not initialize_mt5():
        return

    clean_old_orders()

    log("==============================================")
    log("XAUUSD Hedge Grid Basket Python Bot STARTED")
    log(f"SYMBOL={SYMBOL}")
    log(f"LOTS={LOTS}")
    log(f"GRID_STEP_PRICE={GRID_STEP_PRICE}")
    log(f"GRID_LEVELS={GRID_LEVELS}")
    log(f"BASKET_TARGET_USD={BASKET_TARGET_USD}")
    log(f"MAGIC={MAGIC}")
    log("==============================================")

    try:
        strategy_loop()

    except KeyboardInterrupt:
        log("Bot stopped by user.")

    except Exception as exc:
        log(f"FATAL ERROR: {exc}")

    finally:
        mt5.shutdown()
        log("MT5 connection closed.")


if __name__ == "__main__":
    main()


[2026-08-10 01:02:20] Connected to MT5 | XAUUSD
[2026-08-10 01:02:20] ==============================================
[2026-08-10 01:02:20] XAUUSD Hedge Grid Basket Python Bot STARTED
[2026-08-10 01:02:20] SYMBOL=XAUUSD
[2026-08-10 01:02:20] LOTS=0.01
[2026-08-10 01:02:20] GRID_STEP_PRICE=0.2
[2026-08-10 01:02:20] GRID_LEVELS=5
[2026-08-10 01:02:20] BASKET_TARGET_USD=1.0
[2026-08-10 01:02:20] MAGIC=26080901
[2026-08-10 01:02:20] ==============================================
[2026-08-10 01:02:20] Creating cycle #1 | Ask=4341.49 Bid=4341.3 Step=0.2
[2026-08-10 01:02:20] BUY STOP failed | retcode=10018 | Market closed
[2026-08-10 01:02:20] BUY STOP failed | retcode=10018 | Market closed
[2026-08-10 01:02:21] BUY STOP failed | retcode=10018 | Market closed
[2026-08-10 01:02:21] BUY STOP failed | retcode=10018 | Market closed
[2026-08-10 01:02:21] BUY STOP failed | retcode=10018 | Market closed
[2026-08-10 01:02:21] SELL STOP failed | retcode=10018 | Market closed
[2026-08-10 01:02:21] SELL